# 03 — Exploratory Data Analysis
## PM2.5 & Weather Relationships — Holiday Farm Fire 2020 vs Cedar Creek Fire 2022

Analyses:
1. Side-by-side time series for both events
2. Smoke vs non-smoke weather comparison (per event)
3. Correlation heatmap
4. PM2.5 vs individual weather variables (both events)
5. Diurnal patterns by event
6. Wind direction analysis

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

SMOKE_THRESHOLD = 35  # µg/m³  (EPA 24-h standard)

EVENT_COLORS = {
    'Holiday Farm Fire 2020': '#d62728',
    'Cedar Creek Fire 2022':  '#1f77b4',
}
EVENTS = list(EVENT_COLORS.keys())

## 0. Load Processed Data

In [ ]:
df = pd.read_csv("../data/processed/analysis_data.csv", parse_dates=["timestamp"])
print(f"Loaded {len(df):,} records")

df['is_smoke'] = df['pm2.5_lrapa'] >= SMOKE_THRESHOLD

# Split by event — use dict for easy iteration
events = {e: df[df['event'] == e].copy() for e in EVENTS if e in df['event'].unique()}

for name, edf in events.items():
    smoke_pct = edf['is_smoke'].mean() * 100
    print(f"\n{name}")
    print(f"  rows: {len(edf):,}  |  "
          f"dates: {edf['timestamp'].min().date()} → {edf['timestamp'].max().date()}")
    print(f"  smoke hours (≥{SMOKE_THRESHOLD} µg/m³): "
          f"{edf['is_smoke'].sum()} ({smoke_pct:.1f}%)")
    print(f"  max PM2.5: {edf['pm2.5_lrapa'].max():.1f} µg/m³")

## 1. Side-by-side time series — both fire events

Left column: Holiday Farm Fire 2020 (red). Right column: Cedar Creek Fire 2022 (blue).

In [ ]:
panels = [
    ('pm2.5_lrapa',            'PM2.5 LRAPA-corrected (µg/m³)'),
    ('pm2.5_lrapa_regulatory', 'PM2.5 Regulatory (µg/m³)'),
    ('temperature_f',          'Temperature (°F)'),
    ('humidity',               'Relative Humidity (%)'),
    ('wind_speed_mph',         'Wind Speed (mph)'),
]

n_rows = len(panels)
n_cols = len(events)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 3.2 * n_rows),
                         sharex='col', sharey='row')

# Peak smoke windows per event
PEAKS = {
    'Holiday Farm Fire 2020': ('2020-09-07', '2020-09-15'),
    'Cedar Creek Fire 2022':  ('2022-08-29', '2022-09-12'),
}

for col_idx, (event_name, edf) in enumerate(events.items()):
    color = EVENT_COLORS[event_name]
    for row_idx, (col, label) in enumerate(panels):
        ax = axes[row_idx, col_idx]
        if col not in edf.columns:
            ax.set_visible(False)
            continue
        ax.fill_between(edf['timestamp'], edf[col], alpha=0.2, color=color)
        ax.plot(edf['timestamp'], edf[col], color=color, linewidth=0.8)
        if 'pm2.5' in col:
            ax.axhline(SMOKE_THRESHOLD, color='orange', linestyle='--',
                       linewidth=1, alpha=0.8)
        # Shade peak window
        p0, p1 = PEAKS.get(event_name, (None, None))
        if p0:
            ax.axvspan(pd.Timestamp(p0), pd.Timestamp(p1),
                       alpha=0.08, color=color)
        if row_idx == 0:
            ax.set_title(event_name, fontsize=11, color=color, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(label, fontsize=9)
    # Format x-axis date labels on bottom row
    axes[-1, col_idx].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    axes[-1, col_idx].xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.setp(axes[-1, col_idx].xaxis.get_majorticklabels(),
             rotation=30, ha='right')

plt.suptitle('Fire Event Comparison — Eugene OR', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/fig_full_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Smoke vs Non-Smoke — per event

Smoke = hours with PM2.5 ≥ 35 µg/m³. Welch's t-test p-value shown on each panel.

In [ ]:
weather_cols = [c for c in ['temperature_f', 'humidity', 'wind_speed_mph', 'pressure_hpa']
                if c in df.columns]

fig, axes = plt.subplots(len(events), len(weather_cols),
                         figsize=(14, 4.5 * len(events)), sharey='col')
if len(events) == 1:
    axes = axes[np.newaxis, :]

for row_idx, (event_name, edf) in enumerate(events.items()):
    color = EVENT_COLORS[event_name]
    smoke_df    = edf[edf['is_smoke']]
    no_smoke_df = edf[~edf['is_smoke']]
    for col_idx, col in enumerate(weather_cols):
        ax = axes[row_idx, col_idx]
        d_smoke    = smoke_df[col].dropna()
        d_no_smoke = no_smoke_df[col].dropna()
        bp = ax.boxplot([d_no_smoke, d_smoke],
                        labels=['Clean', 'Smoke'],
                        patch_artist=True,
                        medianprops=dict(color='black', linewidth=2))
        bp['boxes'][0].set_facecolor('lightgray')
        bp['boxes'][1].set_facecolor(color)
        bp['boxes'][1].set_alpha(0.6)
        t, p = stats.ttest_ind(d_smoke, d_no_smoke, equal_var=False)
        ax.set_title(f'{col}\np = {p:.3f}', fontsize=9)
        if col_idx == 0:
            ax.set_ylabel(event_name, fontsize=9, color=color, fontweight='bold')

plt.suptitle('Weather: Smoke vs Non-Smoke Hours (per event)', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/fig_smoke_vs_nonsmoke.png', dpi=150, bbox_inches='tight')
plt.show()

# Print means
print('Mean weather values by event and smoke status:')
for event_name, edf in events.items():
    print(f'\n  {event_name}')
    tbl = edf.groupby('is_smoke')[weather_cols].mean().round(1)
    tbl.index = ['Clean', 'Smoke']
    print(tbl.to_string())

## 3. Correlation Heatmap (each event separately)

In [ ]:
corr_cols = ['pm2.5_lrapa', 'pm2.5_lrapa_regulatory',
             'temperature_f', 'humidity', 'wind_speed_mph',
             'wind_direction', 'pressure_hpa', 'precipitation_in']

fig, axes = plt.subplots(1, len(events), figsize=(9 * len(events), 8))
if len(events) == 1:
    axes = [axes]

for ax, (event_name, edf) in zip(axes, events.items()):
    cols = [c for c in corr_cols if c in edf.columns]
    corr = edf[cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax, square=True,
                linewidths=0.5, cbar_kws={'shrink': 0.8})
    ax.set_title(event_name, fontsize=11)

plt.suptitle('Pearson Correlation Matrix — PM2.5 & Weather', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/fig_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('PM2.5 correlations by event:')
for event_name, edf in events.items():
    cols = [c for c in corr_cols if c in edf.columns]
    corr = edf[cols].corr()
    print(f'\n  {event_name}')
    print(corr['pm2.5_lrapa'].drop('pm2.5_lrapa').sort_values().round(3).to_string())

## 4. PM2.5 vs Individual Weather Variables (both events overlaid)

In [ ]:
scatter_cols = [c for c in ['temperature_f','humidity','wind_speed_mph','pressure_hpa']
                if c in df.columns]
fig, axes = plt.subplots(1, len(scatter_cols), figsize=(14, 4))

for ax, col in zip(axes, scatter_cols):
    for event_name, edf in events.items():
        color = EVENT_COLORS[event_name]
        clean = edf[['pm2.5_lrapa', col]].dropna()
        r, p = stats.pearsonr(clean['pm2.5_lrapa'], clean[col])
        ax.scatter(clean[col], clean['pm2.5_lrapa'],
                   s=5, alpha=0.25, color=color, label=f"{event_name[:4]}… r={r:.2f}")
    ax.set_xlabel(col)
    ax.set_ylabel('PM2.5 LRAPA (µg/m³)' if ax == axes[0] else '')
    ax.legend(fontsize=7)

plt.suptitle('PM2.5 vs Weather Variables', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/fig_pm25_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Diurnal Patterns by Event

In [ ]:
diurnal_cols = [c for c in ['pm2.5_lrapa','temperature_f','humidity','wind_speed_mph']
                if c in df.columns]
fig, axes = plt.subplots(1, len(diurnal_cols), figsize=(14, 4))

for ax, col in zip(axes, diurnal_cols):
    for event_name, edf in events.items():
        color = EVENT_COLORS[event_name]
        hourly = edf.groupby('hour')[col].agg(['mean','sem']).reset_index()
        ax.fill_between(hourly['hour'],
                        hourly['mean'] - hourly['sem'],
                        hourly['mean'] + hourly['sem'],
                        alpha=0.2, color=color)
        ax.plot(hourly['hour'], hourly['mean'], color=color,
                linewidth=2, label=event_name.split()[2] + ' ' + event_name.split()[3])
    ax.set_xlabel('Hour of day (Pacific)')
    ax.set_ylabel(col)
    ax.set_xticks(range(0, 24, 4))
    ax.legend(fontsize=8)

plt.suptitle('Diurnal Patterns — Mean ± SE by Hour (both events)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/fig_diurnal.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Wind Direction Analysis

In [ ]:
if 'wind_direction' in df.columns:
    fig, axes = plt.subplots(1, len(events), figsize=(7 * len(events), 4),
                             subplot_kw=dict(projection='polar'))
    if len(events) == 1:
        axes = [axes]
    for ax, (event_name, edf) in zip(axes, events.items()):
        color = EVENT_COLORS[event_name]
        wd = edf[['wind_direction','pm2.5_lrapa','wind_speed_mph']].dropna()
        bins = np.arange(0, 361, 22.5)
        counts, _ = np.histogram(wd['wind_direction'], bins=bins)
        smoke_counts, _ = np.histogram(
            wd.loc[wd['pm2.5_lrapa'] >= SMOKE_THRESHOLD, 'wind_direction'], bins=bins)
        theta = np.deg2rad((bins[:-1] + bins[1:]) / 2)
        width = np.deg2rad(22.5)
        ax.bar(theta, counts, width=width, alpha=0.4, color='gray', label='All hours')
        ax.bar(theta, smoke_counts, width=width, alpha=0.7, color=color,
               label=f'Smoke ≥{SMOKE_THRESHOLD}')
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)
        ax.set_title(event_name, pad=15, fontsize=10, color=color)
        ax.legend(loc='lower right', fontsize=8)
    plt.suptitle('Wind Direction — All Hours vs Smoke Hours', fontsize=12)
    plt.tight_layout()
    plt.savefig('../data/processed/fig_wind_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('wind_direction column not found — skipping.')